# Tokenization — Build: Train & Evaluate a Domain Tokenizer

> **Section 01 — Topic 01 — build**

**Prerequisites:** All tokenization notebooks (foundations, deep-dive)

---

In this notebook we go from theory to practice. You will:

1. **Prepare a real corpus** — Python source code from open-source projects on GitHub
2. **Train three tokenizers from scratch** — BPE, Unigram, and byte-level BPE — using the HuggingFace `tokenizers` library
3. **Evaluate quantitatively** — fertility, compression ratio, unknown-token rate, speed
4. **Compare against production tokenizers** — GPT-2 and Llama
5. **Analyze qualitatively** — side-by-side tokenizations, edge cases, code-structure preservation
6. **Package and export** — save the best tokenizer in HuggingFace format with a tokenizer card

By the end you will have a fully trained, evaluated, and packaged domain-specific tokenizer ready for downstream use.

## Setup

Install the required libraries. We need:
- `tokenizers` — the fast Rust-backed tokenizer training library from HuggingFace
- `transformers` — to load pre-trained tokenizers (GPT-2, Llama) for comparison
- `datasets` — to download real Python source code from the HuggingFace Hub
- `matplotlib` — for visualization

In [ ]:
!pip install -q tokenizers transformers datasets matplotlib

In [ ]:
import os
import time
import json
import textwrap
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers, decoders
from tokenizers.processors import TemplateProcessing
from transformers import AutoTokenizer

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 12,
})

OUTPUT_DIR = Path("./tokenizer_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Setup complete.")

---

## 1. Corpus Preparation

A tokenizer is only as good as its training corpus. Since we are building a **code-domain** tokenizer,
we need a large, representative sample of Python source code.

We will use the `codeparrot/github-code` dataset on the HuggingFace Hub, filtering for Python files.
This gives us real-world code with all the diversity that entails — docstrings, comments,
complex expressions, unicode identifiers, and more.

### Why corpus composition matters

The merge rules a BPE tokenizer learns are entirely determined by substring frequencies in the
training corpus. If your corpus is heavily skewed (e.g., all scientific computing), the tokenizer
will over-specialize — `numpy` might become a single token while `flask` stays fragmented.
A balanced, diverse corpus produces a tokenizer that generalizes well across the domain.

In [ ]:
# Download Python source code from codeparrot/github-code
# We stream to avoid downloading the entire dataset (which is hundreds of GB).
# We'll collect a manageable subset for training.

NUM_TRAIN_FILES = 20_000
NUM_EVAL_FILES = 2_000

print(f"Streaming Python files from codeparrot/github-code ...")
print(f"  Train: {NUM_TRAIN_FILES:,} files")
print(f"  Eval:  {NUM_EVAL_FILES:,} files")

ds = load_dataset(
    "codeparrot/github-code",
    streaming=True,
    split="train",
    languages=["Python"],
    licenses=["mit"],
)

train_texts = []
eval_texts = []

for i, example in enumerate(ds):
    if i >= NUM_TRAIN_FILES + NUM_EVAL_FILES:
        break
    text = example["code"]
    # Skip very short or very long files
    if len(text) < 100 or len(text) > 50_000:
        continue
    if i < NUM_TRAIN_FILES:
        train_texts.append(text)
    else:
        eval_texts.append(text)

print(f"\nCollected {len(train_texts):,} training files and {len(eval_texts):,} eval files.")

In [ ]:
# Corpus statistics
train_lengths = [len(t) for t in train_texts]
eval_lengths = [len(t) for t in eval_texts]

total_train_chars = sum(train_lengths)
total_eval_chars = sum(eval_lengths)

print("=== Corpus Statistics ===")
print(f"Training set: {len(train_texts):,} files, {total_train_chars:,} characters")
print(f"  Mean file length: {np.mean(train_lengths):,.0f} chars")
print(f"  Median file length: {np.median(train_lengths):,.0f} chars")
print(f"  Min / Max: {min(train_lengths):,} / {max(train_lengths):,} chars")
print(f"\nEval set: {len(eval_texts):,} files, {total_eval_chars:,} characters")
print(f"  Mean file length: {np.mean(eval_lengths):,.0f} chars")

In [ ]:
# Visualize the distribution of file lengths
fig, ax = plt.subplots()
ax.hist(train_lengths, bins=80, color="steelblue", edgecolor="white", alpha=0.85)
ax.set_xlabel("File length (characters)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Python file lengths in training corpus")
ax.axvline(np.median(train_lengths), color="tomato", ls="--", label=f"Median = {np.median(train_lengths):,.0f}")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save training texts to disk so the tokenizer trainer can read from files
# (the tokenizers library trains most efficiently from files)
TRAIN_FILE = OUTPUT_DIR / "train_corpus.txt"

with open(TRAIN_FILE, "w", encoding="utf-8") as f:
    for text in train_texts:
        f.write(text + "\n\n")

print(f"Training corpus written to {TRAIN_FILE}")
print(f"File size: {TRAIN_FILE.stat().st_size / 1e6:.1f} MB")

---

## 2. Train Three Tokenizers

We will train three tokenizers, all with `vocab_size=32000`, to compare the three major
subword algorithms head-to-head on the same corpus:

| Tokenizer | Algorithm | Key idea |
|-----------|-----------|----------|
| **BPE** | Byte-Pair Encoding | Bottom-up: iteratively merge the most frequent pair of adjacent tokens |
| **Unigram** | Unigram Language Model | Top-down: start with a large vocabulary and prune tokens that least reduce the corpus likelihood |
| **Byte-level BPE** | BPE on raw bytes | Like BPE but operates on byte sequences — no unknown tokens possible |

All three use the HuggingFace `tokenizers` library, which is written in Rust and trains
on millions of lines in seconds.

### 2.1 BPE Tokenizer

Classic BPE starts with a character-level vocabulary and iteratively merges the most frequent
bigram. After `vocab_size` merges, you have a vocabulary of subword units tuned to your corpus.

For code, we use a pre-tokenizer that splits on whitespace and punctuation boundaries —
this prevents merges from crossing syntactic boundaries (e.g., we don't want `):` to become
a single token just because it's frequent).

In [ ]:
VOCAB_SIZE = 32_000
SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]

# --- BPE Tokenizer ---
bpe_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

bpe_tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFC(),          # Unicode normalization
    normalizers.Replace("\r\n", "\n"),  # Normalize line endings
])

bpe_tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.WhitespaceSplit(),
    pre_tokenizers.Punctuation(),
])

bpe_trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    min_frequency=2,
    show_progress=True,
)

print("Training BPE tokenizer...")
t0 = time.perf_counter()
bpe_tokenizer.train([str(TRAIN_FILE)], trainer=bpe_trainer)
bpe_time = time.perf_counter() - t0

print(f"BPE training complete in {bpe_time:.1f}s")
print(f"Vocabulary size: {bpe_tokenizer.get_vocab_size():,}")

### 2.2 Unigram Tokenizer

Unigram takes the opposite approach: start with a very large vocabulary (often 10x the target)
and iteratively remove tokens whose removal least increases the overall corpus loss under a
unigram language model. The result is a vocabulary where each token has a learned log-probability,
and tokenization is done via the Viterbi algorithm (finding the most probable segmentation).

Unigram tends to produce more linguistically motivated subwords and handles rare words more
gracefully than BPE.

In [ ]:
# --- Unigram Tokenizer ---
unigram_tokenizer = Tokenizer(models.Unigram())

unigram_tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFC(),
    normalizers.Replace("\r\n", "\n"),
])

unigram_tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.WhitespaceSplit(),
    pre_tokenizers.Punctuation(),
])

unigram_trainer = trainers.UnigramTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    unk_token="<unk>",
    show_progress=True,
)

print("Training Unigram tokenizer...")
t0 = time.perf_counter()
unigram_tokenizer.train([str(TRAIN_FILE)], trainer=unigram_trainer)
unigram_time = time.perf_counter() - t0

print(f"Unigram training complete in {unigram_time:.1f}s")
print(f"Vocabulary size: {unigram_tokenizer.get_vocab_size():,}")

### 2.3 Byte-Level BPE Tokenizer

Byte-level BPE (used by GPT-2, RoBERTa, and many modern models) operates on raw byte
sequences rather than Unicode characters. Every possible input can be represented — there
are no unknown tokens.

The trick: map each of the 256 byte values to a printable Unicode character, run BPE on
those, then decode back to bytes. This gives you the efficiency of BPE with the universality
of byte-level representation.

In [ ]:
# --- Byte-Level BPE Tokenizer ---
byte_bpe_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

# No normalizer — byte-level BPE works on raw bytes
byte_bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
byte_bpe_tokenizer.decoder = decoders.ByteLevel()

byte_bpe_trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
    min_frequency=2,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    show_progress=True,
)

print("Training Byte-Level BPE tokenizer...")
t0 = time.perf_counter()
byte_bpe_tokenizer.train([str(TRAIN_FILE)], trainer=byte_bpe_trainer)
byte_bpe_time = time.perf_counter() - t0

print(f"Byte-Level BPE training complete in {byte_bpe_time:.1f}s")
print(f"Vocabulary size: {byte_bpe_tokenizer.get_vocab_size():,}")

In [ ]:
# Training time comparison
print("=== Training Time Summary ===")
print(f"  BPE:            {bpe_time:.1f}s")
print(f"  Unigram:        {unigram_time:.1f}s")
print(f"  Byte-Level BPE: {byte_bpe_time:.1f}s")

---

## 3. Comparative Evaluation

Now we evaluate all three custom tokenizers **and** two production tokenizers (GPT-2 and Llama)
on the same held-out eval set. We measure:

| Metric | Definition | Why it matters |
|--------|-----------|----------------|
| **Fertility** | Average number of tokens per word | Lower = more efficient encoding |
| **Compression ratio** | Characters per token | Higher = fewer tokens needed to represent text |
| **Unknown token rate** | Fraction of tokens that are `<unk>` | Lower = better coverage |
| **Tokenization speed** | Characters per second | Higher = faster inference preprocessing |

In [ ]:
# Load production tokenizers for comparison
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
print(f"GPT-2 tokenizer loaded. Vocab size: {gpt2_tokenizer.vocab_size:,}")

# For Llama, we use the open tokenizer from meta-llama
# If you don't have access, this will fall back gracefully
try:
    llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
    print(f"Llama-2 tokenizer loaded. Vocab size: {llama_tokenizer.vocab_size:,}")
    HAS_LLAMA = True
except Exception as e:
    print(f"Could not load Llama tokenizer ({e}). Will skip Llama comparisons.")
    llama_tokenizer = None
    HAS_LLAMA = False

In [ ]:
def evaluate_hf_tokenizer(tokenizer, texts, name):
    """Evaluate a HuggingFace transformers tokenizer on a list of texts."""
    total_tokens = 0
    total_chars = 0
    total_words = 0
    total_unks = 0
    unk_id = getattr(tokenizer, "unk_token_id", None)

    t0 = time.perf_counter()
    for text in texts:
        ids = tokenizer.encode(text)
        total_tokens += len(ids)
        total_chars += len(text)
        total_words += len(text.split())
        if unk_id is not None:
            total_unks += ids.count(unk_id) if isinstance(ids, list) else 0
    elapsed = time.perf_counter() - t0

    return {
        "name": name,
        "fertility": total_tokens / total_words if total_words else 0,
        "compression": total_chars / total_tokens if total_tokens else 0,
        "unk_rate": total_unks / total_tokens if total_tokens else 0,
        "speed_kchars_s": (total_chars / 1000) / elapsed if elapsed else 0,
        "total_tokens": total_tokens,
    }


def evaluate_tokenizers_lib(tokenizer, texts, name):
    """Evaluate a tokenizers-library tokenizer on a list of texts."""
    total_tokens = 0
    total_chars = 0
    total_words = 0
    total_unks = 0
    unk_id = tokenizer.token_to_id("<unk>")

    t0 = time.perf_counter()
    for text in texts:
        encoding = tokenizer.encode(text)
        ids = encoding.ids
        total_tokens += len(ids)
        total_chars += len(text)
        total_words += len(text.split())
        if unk_id is not None:
            total_unks += ids.count(unk_id)
    elapsed = time.perf_counter() - t0

    return {
        "name": name,
        "fertility": total_tokens / total_words if total_words else 0,
        "compression": total_chars / total_tokens if total_tokens else 0,
        "unk_rate": total_unks / total_tokens if total_tokens else 0,
        "speed_kchars_s": (total_chars / 1000) / elapsed if elapsed else 0,
        "total_tokens": total_tokens,
    }


print("Evaluation functions defined.")

In [ ]:
# Run evaluation on the held-out eval set
print("Evaluating all tokenizers on the eval set...\n")

results = []
results.append(evaluate_tokenizers_lib(bpe_tokenizer, eval_texts, "BPE (ours)"))
results.append(evaluate_tokenizers_lib(unigram_tokenizer, eval_texts, "Unigram (ours)"))
results.append(evaluate_tokenizers_lib(byte_bpe_tokenizer, eval_texts, "Byte-BPE (ours)"))
results.append(evaluate_hf_tokenizer(gpt2_tokenizer, eval_texts, "GPT-2"))
if HAS_LLAMA:
    results.append(evaluate_hf_tokenizer(llama_tokenizer, eval_texts, "Llama-2"))

# Print results table
header = f"{'Tokenizer':<18} {'Fertility':>10} {'Compression':>12} {'UNK Rate':>10} {'Speed (kc/s)':>13} {'Total Tokens':>13}"
print(header)
print("-" * len(header))
for r in results:
    print(
        f"{r['name']:<18} {r['fertility']:>10.3f} {r['compression']:>12.3f} "
        f"{r['unk_rate']:>10.5f} {r['speed_kchars_s']:>13,.0f} {r['total_tokens']:>13,}"
    )

In [ ]:
# Visualization: Compression ratio and Fertility comparison
names = [r["name"] for r in results]
compressions = [r["compression"] for r in results]
fertilities = [r["fertility"] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"][:len(results)]

# Compression ratio
axes[0].barh(names, compressions, color=colors, edgecolor="white")
axes[0].set_xlabel("Characters per token (higher = more efficient)")
axes[0].set_title("Compression Ratio")
for i, v in enumerate(compressions):
    axes[0].text(v + 0.05, i, f"{v:.2f}", va="center", fontsize=11)

# Fertility
axes[1].barh(names, fertilities, color=colors, edgecolor="white")
axes[1].set_xlabel("Tokens per word (lower = more efficient)")
axes[1].set_title("Fertility")
for i, v in enumerate(fertilities):
    axes[1].text(v + 0.02, i, f"{v:.2f}", va="center", fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "compression_fertility.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to", OUTPUT_DIR / "compression_fertility.png")

In [ ]:
# Visualization: Tokenization speed
speeds = [r["speed_kchars_s"] for r in results]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(names, speeds, color=colors, edgecolor="white")
ax.set_xlabel("Thousand characters per second")
ax.set_title("Tokenization Speed")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
for i, v in enumerate(speeds):
    ax.text(v + max(speeds) * 0.01, i, f"{v:,.0f}", va="center", fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tokenization_speed.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to", OUTPUT_DIR / "tokenization_speed.png")

In [ ]:
# Visualization: Unknown token rate
unk_rates = [r["unk_rate"] * 100 for r in results]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(names, unk_rates, color=colors, edgecolor="white")
ax.set_xlabel("Unknown token rate (%)")
ax.set_title("Unknown Token Rate (lower is better)")
for i, v in enumerate(unk_rates):
    ax.text(v + max(max(unk_rates), 0.001) * 0.02, i, f"{v:.4f}%", va="center", fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "unk_rate.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpreting the results

**Compression ratio:** Our domain-specific tokenizers should achieve a *higher* compression ratio on
Python code than GPT-2 or Llama, because they were trained specifically on this distribution.
A general-purpose tokenizer wastes vocabulary capacity on tokens from natural language, HTML,
and other domains it will never see in our use case.

**Fertility:** Closely related to compression. A domain tokenizer should produce fewer tokens
per whitespace-delimited word, meaning less work for the downstream model.

**Unknown token rate:** Byte-level BPE should have 0% unknowns (by design). Character-level
BPE and Unigram may have a small unknown rate if they encounter Unicode characters not seen
in training.

**Speed:** The `tokenizers` library tokenizers (Rust-backed) should be significantly faster than
the HuggingFace `transformers` wrappers, which add Python overhead.